# I'm gonna run a test model here before diving deep into the ultimate model

## I. Introduction
This document only focus on a simple ML model using lightgbm to test the dataset.
I will note my progress making and trying simple model (only use libraries and modules) step-by-step.


## II, Tutorial
### Step 1: Import library.

In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

### Step 2:Prepare the dataset

In [8]:
file_path = r'G:\url-analysis\src\data\malicious_phish_processed.csv'
whitelist_path = r'G:\url-analysis\src\data\domain.txt'
url_data = pd.read_csv(file_path, encoding_errors='ignore')
url_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 653490 entries, 0 to 653489
Data columns (total 65 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   url                               653490 non-null  str    
 1   type                              653490 non-null  str    
 2   url length                        653490 non-null  int64  
 3   number_of_part                    653490 non-null  int64  
 4   has_scheme                        653490 non-null  int64  
 5   has_netloc                        653490 non-null  int64  
 6   has_path                          653490 non-null  int64  
 7   has_params                        653490 non-null  int64  
 8   has_query                         653490 non-null  int64  
 9   has_fragment                      653490 non-null  int64  
 10  has_username                      653490 non-null  int64  
 11  has_password                      653490 non-null  int64  
 12 

### Step 3: split the dataset, X is the set of features used to test, y is the label

In [3]:
X = pd.DataFrame(url_data).drop(columns = ['url', 'type'])
print(X.head())

# y is the tag for supervised learning
y = url_data.type
print(y)

   url length  number_of_part  has_scheme  has_netloc  has_path  has_params  \
0          16               1           0           1         0           0   
1          35               2           0           1         1           0   
2          31               2           0           1         1           0   
3          88               4           1           1         1           0   
4         235               4           1           1         1           0   

   has_query  has_fragment  has_username  has_password  ...  has_uuid_path  \
0          0             0             0             0  ...              0   
1          0             0             0             0  ...              0   
2          0             0             0             0  ...              0   
3          1             0             0             0  ...              0   
4          1             0             0             0  ...              0   

   download_param  free_host  free_host_download  suspic

### Step 4: split X, y to X_train, X_test, y_train, y_test to train and test using LightGBM model

In [4]:
le = LabelEncoder()
encoded_y = le.fit_transform(y)
print(encoded_y)


X_train, X_test, y_train, y_test = train_test_split(X, encoded_y, test_size = 0.4, random_state = 32)
train_data = lgb.Dataset(X_train, label = y_train)
test_data = lgb.Dataset(X_test, label = y_test)

params = {
    'objective' : 'multiclass',
    'num_class' : 4,
    'metric' : ['multi_logloss', 'multi_error', 'auc_mu'],
    'boosting_type' : 'gbdt',
    'learning_rate' : 0.05,
    'verbose' : 0,
}

model_classification = lgb.train(params, train_data, valid_sets = [test_data], num_boost_round = 2000 )

[3 0 0 ... 3 3 3]
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines


In [5]:
y_pred = model_classification.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis = 1)


print("=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===")
accuracy = accuracy_score(y_test, y_pred_classes)
print(f"Độ chính xác (Accuracy): {accuracy:.4f}\n")

# In ra Classification Report chi tiết (Precision, Recall, F1-score)
# Sử dụng le.classes_ để map ngược lại tên chuỗi ban đầu (phishing, benign, malware, defacement)
print("Báo cáo phân loại chi tiết:")
print(classification_report(y_test, y_pred_classes, target_names=le.classes_))

# 3. Lưu mô hình (vì ở step 2 bạn đã import joblib)
model_filename = 'lgb_url_classifier.pkl'
joblib.dump(model_classification, model_filename)
print(f"Đã lưu mô hình thành công tại: {model_filename}")

=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===
Độ chính xác (Accuracy): 0.9754

Báo cáo phân loại chi tiết:
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99    171228
  defacement       0.98      1.00      0.99     38778
     malware       0.99      0.95      0.97     12788
    phishing       0.94      0.90      0.92     38602

    accuracy                           0.98    261396
   macro avg       0.97      0.96      0.96    261396
weighted avg       0.98      0.98      0.98    261396

Đã lưu mô hình thành công tại: lgb_url_classifier.pkl
